# 1. 데이터 준비
## 건설현장 안전장비 감지 모델 학습을 위한 데이터 준비

이 노트북에서는 다음을 수행합니다:
1. 데이터셋 다운로드 (공개 데이터셋 활용)
2. 데이터 탐색 및 분석
3. 데이터 전처리 및 증강
4. S3 업로드

In [ ]:
# 필요한 라이브러리 설치
!pip install -q ultralytics boto3 sagemaker pandas matplotlib

In [ ]:
import os
import json
import shutil
from pathlib import Path

import boto3
import sagemaker
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# SageMaker 세션
session = sagemaker.Session()
bucket = session.default_bucket()
prefix = 'construction-safety'
region = session.boto_region_name

print(f"S3 버킷: {bucket}")
print(f"리전: {region}")

## 1.1 데이터셋 다운로드

공개 안전장비 데이터셋을 활용합니다.
- Safety Helmet Detection Dataset
- Construction Site Safety Dataset

In [ ]:
# 데이터 디렉토리 생성
DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

(DATA_DIR / 'train' / 'images').mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'train' / 'labels').mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'val' / 'images').mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'val' / 'labels').mkdir(parents=True, exist_ok=True)

print("데이터 디렉토리 구조 생성 완료")

In [ ]:
# Roboflow에서 데이터셋 다운로드 (예시)
# 실제 사용 시 API 키 필요

# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace().project("construction-safety")
# dataset = project.version(1).download("yolov8")

# 또는 직접 다운로드
print("데이터셋을 다운로드하려면 Roboflow 또는 다른 소스를 사용하세요.")
print("예시 데이터셋:")
print("- https://universe.roboflow.com/roboflow-100/hard-hat-workers")
print("- https://universe.roboflow.com/ppe-detection-yolov8/safety-helmet-detection")

## 1.2 데이터 탐색

In [ ]:
def count_dataset_stats(data_dir):
    """데이터셋 통계 계산"""
    stats = {
        'train_images': 0,
        'val_images': 0,
        'class_distribution': {}
    }
    
    # 클래스 이름
    classes = ['person', 'hardhat', 'safety_vest', 'safety_shoes', 'no_hardhat', 'no_safety_vest']
    
    for split in ['train', 'val']:
        images_dir = data_dir / split / 'images'
        labels_dir = data_dir / split / 'labels'
        
        if images_dir.exists():
            image_count = len(list(images_dir.glob('*.jpg'))) + len(list(images_dir.glob('*.png')))
            stats[f'{split}_images'] = image_count
            
            # 라벨 분석
            for label_file in labels_dir.glob('*.txt'):
                with open(label_file) as f:
                    for line in f:
                        class_id = int(line.strip().split()[0])
                        class_name = classes[class_id] if class_id < len(classes) else 'unknown'
                        stats['class_distribution'][class_name] = stats['class_distribution'].get(class_name, 0) + 1
    
    return stats

# 통계 출력
if DATA_DIR.exists():
    stats = count_dataset_stats(DATA_DIR)
    print("=== 데이터셋 통계 ===")
    print(f"학습 이미지: {stats['train_images']}")
    print(f"검증 이미지: {stats['val_images']}")
    print(f"\n클래스 분포:")
    for cls, count in stats['class_distribution'].items():
        print(f"  {cls}: {count}")

In [ ]:
def visualize_samples(data_dir, num_samples=6):
    """샘플 이미지 시각화"""
    images_dir = data_dir / 'train' / 'images'
    labels_dir = data_dir / 'train' / 'labels'
    
    classes = ['person', 'hardhat', 'safety_vest', 'safety_shoes', 'no_hardhat', 'no_safety_vest']
    colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255), (0, 255, 255)]
    
    image_files = list(images_dir.glob('*.jpg'))[:num_samples]
    
    if not image_files:
        print("이미지 파일이 없습니다.")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(image_files):
        img = np.array(Image.open(img_path))
        h, w = img.shape[:2]
        
        # 라벨 로드 및 박스 그리기
        label_path = labels_dir / f"{img_path.stem}.txt"
        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    class_id = int(parts[0])
                    x_center, y_center, box_w, box_h = map(float, parts[1:5])
                    
                    # 좌표 변환
                    x1 = int((x_center - box_w/2) * w)
                    y1 = int((y_center - box_h/2) * h)
                    x2 = int((x_center + box_w/2) * w)
                    y2 = int((y_center + box_h/2) * h)
                    
                    # 박스 그리기
                    import cv2
                    color = colors[class_id % len(colors)]
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(img, classes[class_id], (x1, y1-5), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        axes[idx].imshow(img)
        axes[idx].set_title(img_path.name)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# 샘플 시각화
# visualize_samples(DATA_DIR)

## 1.3 데이터 증강

In [ ]:
import albumentations as A

# 데이터 증강 파이프라인
augmentation = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.RandomShadow(p=0.2),
    A.GaussNoise(p=0.2),
    A.MotionBlur(blur_limit=3, p=0.1),
    A.CLAHE(p=0.2),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

print("데이터 증강 파이프라인 정의됨")
print("증강 기법:")
print("- 수평 뒤집기")
print("- 밝기/대비 조절")
print("- 랜덤 그림자")
print("- 가우시안 노이즈")
print("- 모션 블러")
print("- CLAHE (대비 제한 적응 히스토그램 균등화)")

## 1.4 S3 업로드

In [ ]:
def upload_to_s3(local_dir, s3_prefix):
    """로컬 디렉토리를 S3에 업로드"""
    s3_client = boto3.client('s3')
    
    for root, dirs, files in os.walk(local_dir):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, local_dir)
            s3_key = f"{s3_prefix}/{relative_path}"
            
            print(f"업로드: {local_path} -> s3://{bucket}/{s3_key}")
            s3_client.upload_file(local_path, bucket, s3_key)
    
    return f"s3://{bucket}/{s3_prefix}"

# 데이터 업로드
# train_s3_path = upload_to_s3(DATA_DIR / 'train', f'{prefix}/data/train')
# val_s3_path = upload_to_s3(DATA_DIR / 'val', f'{prefix}/data/val')

print("\n데이터 업로드 준비 완료")
print(f"학습 데이터 경로: s3://{bucket}/{prefix}/data/train")
print(f"검증 데이터 경로: s3://{bucket}/{prefix}/data/val")

## 1.5 데이터 설정 파일 생성

In [ ]:
# YOLOv8 데이터 설정 파일
data_config = {
    'path': f's3://{bucket}/{prefix}/data',
    'train': 'train/images',
    'val': 'val/images',
    'names': {
        0: 'person',
        1: 'hardhat',
        2: 'safety_vest',
        3: 'safety_shoes',
        4: 'no_hardhat',
        5: 'no_safety_vest'
    },
    'nc': 6
}

print("데이터 설정:")
print(json.dumps(data_config, indent=2, ensure_ascii=False))

## 다음 단계

데이터 준비가 완료되었습니다. 다음 노트북에서 모델 학습을 진행합니다.

→ `02_model_training.ipynb`